# Validação 10 — Extração de afirmações científicas

## Goal

Comprovar que o ranking híbrido pode ser transformado em afirmações científicas curtas, sem duplicação e com proveniência suficiente para a futura comparação com a alegação do usuário.

## Setup

A extração separa sentenças dos trechos recuperados, prioriza cobertura de termos da alegação e posição híbrida, e limita a quantidade por trecho. Ela não classifica apoio ou contradição: todos os pares permanecem com status `PENDING`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    DEFAULT_EMBEDDING_MODEL,
    Bm25Index,
    ChunkingConfig,
    ExtractionConfig,
    HybridIndex,
    PmcClient,
    PubMedClient,
    SemanticIndex,
    SentenceTransformerEncoder,
    build_claim_evidence_pairs,
    chunk_article_content,
    extract_evidence_statements,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-25T11:34:36.113457+00:00


## Steps

### 1. Recuperar as evidências híbridas

O fluxo é reexecutado desde as fontes oficiais para que a extração seja comprovada sobre conteúdo real.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

claim = "Beber café pode alterar o risco de câncer de próstata."
analysis_input = validate_analysis_input(claim)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv("NCBI_EMAIL"), api_key=os.getenv("NCBI_API_KEY")),
    max_results_per_query=1,
)
content = retrieve_article_content(
    pubmed_result.publications[0],
    PmcClient(email=os.getenv("NCBI_EMAIL"), api_key=os.getenv("NCBI_API_KEY")),
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=120, overlap_words=20))
lexical_index = Bm25Index(chunks)
semantic_index = SemanticIndex(
    chunks,
    SentenceTransformerEncoder(os.getenv("EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)),
)
hybrid_results = HybridIndex(lexical_index, semantic_index).search(claim, top_k=8)
print(f"Fonte: {content.pmcid} | trechos híbridos usados: {len(hybrid_results)}")

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22974.58it/s]

Fonte: PMC7805365 | trechos híbridos usados: 8


### 2. Extrair afirmações e criar pares

A sobreposição lexical atua apenas como sinal transparente de seleção; a relevância bilíngue já foi estabelecida pelo ranking híbrido.

In [3]:
extraction_config = ExtractionConfig(
    max_statements=6,
    max_per_chunk=2,
    min_words=8,
    max_words=80,
)
statements = extract_evidence_statements(claim, hybrid_results, extraction_config)
pairs = build_claim_evidence_pairs(claim, statements)

print(f"Afirmações extraídas: {len(statements)} | pares preparados: {len(pairs)}")

Afirmações extraídas: 6 | pares preparados: 6


In [4]:
rows = [
    {
        "statement_id": statement.statement_id,
        "extraction_score": round(statement.extraction_score, 4),
        "matched_claim_terms": statement.matched_claim_terms,
        "hybrid_rank": statement.hybrid_rank,
        "section": statement.section,
        "statement": statement.text,
        "pmid": statement.pmid,
        "pmcid": statement.pmcid,
        "source_url": statement.source_url,
        "assessment_status": pair.assessment_status,
    }
    for statement, pair in zip(statements, pairs)
]
pprint(rows)

[{'assessment_status': 'PENDING',
  'extraction_score': 0.3936,
  'hybrid_rank': 3,
  'matched_claim_terms': ('cancer',),
  'pmcid': 'PMC7805365',
  'pmid': '33431520',
  'section': 'Conclusions',
  'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/',
  'statement': 'This study suggests that increased coffee consumption may be '
               'associated with a reduced risk of prostate cancer.',
  'statement_id': 'statement:3ebd0e964bc9c8e0'},
 {'assessment_status': 'PENDING',
  'extraction_score': 0.3455,
  'hybrid_rank': 7,
  'matched_claim_terms': ('cancer',),
  'pmcid': 'PMC7805365',
  'pmid': '33431520',
  'section': 'Results',
  'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/',
  'statement': 'a linear inverse association between coffee consumption and '
               'prostate cancer risk (p=0.006 for linear trend) (figure 3).',
  'statement_id': 'statement:eeb6f13090c99cde'},
 {'assessment_status': 'PENDING',
  'extraction_score': 0.3455,
  'hyb

## Checks

As verificações confirmam limites, ordenação, ausência de duplicatas, identificadores estáveis, proveniência e ausência de classificação prematura.

In [5]:
assert 1 <= len(statements) <= extraction_config.max_statements
assert len(pairs) == len(statements)
assert len({statement.statement_id for statement in statements}) == len(statements)
assert len({statement.text.casefold() for statement in statements}) == len(statements)
assert all(
    previous.extraction_score >= current.extraction_score
    for previous, current in zip(statements, statements[1:])
)
assert all(statement.matched_claim_terms for statement in statements)
assert all(statement.pmid == "33431520" for statement in statements)
assert all(statement.pmcid == "PMC7805365" for statement in statements)
assert all(statement.source_url == content.pmc_url for statement in statements)
assert all(pair.assessment_status == "PENDING" for pair in pairs)
assert all(pair.claim == claim for pair in pairs)
assert any("prostate cancer" in statement.text.casefold() for statement in statements)

second_run = extract_evidence_statements(claim, hybrid_results, extraction_config)
assert [statement.statement_id for statement in statements] == [
    statement.statement_id for statement in second_run
]

print(
    f"Validação aprovada: {len(statements)} afirmações rastreáveis; "
    "nenhuma relação foi classificada antes da próxima etapa."
)

Validação aprovada: 6 afirmações rastreáveis; nenhuma relação foi classificada antes da próxima etapa.


## Next Steps

A extração estará validada quando todas as células forem executadas sem erros. A próxima etapa será classificar cada par como apoio, contradição ou evidência neutra, mantendo incerteza e justificativa.